In [7]:
import pandas as pd
import numpy as numpy

# EDA Process

In [8]:
#Load Dataset
df = pd.read_csv("BNPL_dataset.csv")
df.head()

,user_id,age,employment_type,monthly_income,credit_score,purchase_amount,product_category,bnpl_installments,repayment_delay_days,missed_payments,default_flag,app_usage_frequency,location,transaction_date,debt_to_income_ratio,risk_score,customer_segment
0,1,56,Salaried,68529.50,552,5000.00,Electronics,12,13,1,0,8.49,Australia,2023-06-10,0.072961,165.2,Medium Risk
1,2,19,Student,7247.85,300,1073.23,Fashion,12,13,1,0,3.09,USA,2024-10-07,0.148076,266.0,High Risk
2,3,20,Self-Employed,41582.26,471,5000.00,Electronics,3,19,2,0,3.33,Australia,2023-04-05,0.120244,229.6,High Risk
3,4,21,Salaried,14423.46,300,4076.83,Sports,6,18,5,1,5.86,Germany,2023-06-24,0.282653,356.0,High Risk
4,5,43,Salaried,42845.50,512,5000.00,Electronics,9,0,0,0,7.36,India,2024-10-19,0.116698,135.2,High Risk


In [9]:
#Check missing values
df.isnull().sum()

user_id                 0
age                     0
employment_type         0
monthly_income          0
credit_score            0
purchase_amount         0
product_category        0
bnpl_installments       0
repayment_delay_days    0
missed_payments         0
default_flag            0
app_usage_frequency     0
location                0
transaction_date        0
debt_to_income_ratio    0
risk_score              0
customer_segment        0
dtype: int64

In [10]:
df.describe()

,user_id,age,monthly_income,credit_score,purchase_amount,bnpl_installments,repayment_delay_days,missed_payments,default_flag,app_usage_frequency,debt_to_income_ratio,risk_score
count,10345.000000,10345.000000,10345.000000,10345.000000,10345.000000,10345.000000,10345.000000,10345.000000,10345.000000,10345.000000,10345.000000,10345.000000
mean,5173.000000,38.559884,35053.381898,448.176124,3979.402721,7.477525,8.742774,1.015950,0.390527,5.524101,0.181071,198.534094
std,2986.488601,12.131789,27084.517277,136.518332,1500.828483,3.362867,6.781849,0.996726,0.487892,2.590364,0.124887,67.541384
min,1.000000,18.000000,5000.000000,300.000000,100.000000,3.000000,0.000000,0.000000,0.000000,1.000000,0.000811,0.000000
25%,2587.000000,28.000000,12207.020000,332.000000,2990.720000,3.000000,2.000000,0.000000,0.000000,3.300000,0.080187,153.600000
50%,5173.000000,39.000000,23800.520000,403.000000,5000.000000,9.000000,9.000000,1.000000,0.000000,5.550000,0.138263,202.800000
75%,7759.000000,49.000000,55276.930000,547.000000,5000.000000,9.000000,14.000000,2.000000,1.000000,7.750000,0.268300,250.800000
max,10345.000000,59.000000,145767.120000,850.000000,5000.000000,12.000000,33.000000,7.000000,1.000000,10.000000,0.723460,398.000000


In [11]:
df.dtypes

user_id                   int64
age                       int64
employment_type          object
monthly_income          float64
credit_score              int64
purchase_amount         float64
product_category         object
bnpl_installments         int64
repayment_delay_days      int64
missed_payments           int64
default_flag              int64
app_usage_frequency     float64
location                 object
transaction_date         object
debt_to_income_ratio    float64
risk_score              float64
customer_segment         object
dtype: object

In [10]:
#One-Hot Encoding
categorical_cols = [
    'employment_type',
    'product_category',
    'location',
    'customer_segment'
]

df = pd.get_dummies(
    df,
    columns=categorical_cols,
    drop_first=True
)

#convert boolean columns to integers
bools_cols=df.select_dtypes(include='bool').columns
df[bools_cols] = df[bools_cols].astype(int)

In [11]:
# Covert Transaction Date to datetime
df['transaction_date']=pd.to_datetime(df['transaction_date'])
df['transaction_month'] = df['transaction_date'].dt.month
df['transaction_year'] = df['transaction_date'].dt.year
df['transaction_day'] = df['transaction_date'].dt.day
df.drop('transaction_date', axis=1, inplace=True)


In [17]:
df.head()

,user_id,age,monthly_income,credit_score,purchase_amount,bnpl_installments,repayment_delay_days,missed_payments,default_flag,app_usage_frequency,...,location_Canada,location_Germany,location_India,location_UK,location_USA,customer_segment_Low Risk,customer_segment_Medium Risk,transaction_month,transaction_year,transaction_day
0,1,56,68529.50,552,5000.00,12,13,1,0,8.49,...,0,0,0,0,0,0,1,6,2023,10
1,2,19,7247.85,300,1073.23,12,13,1,0,3.09,...,0,0,0,0,1,0,0,10,2024,7
2,3,20,41582.26,471,5000.00,3,19,2,0,3.33,...,0,0,0,0,0,0,0,4,2023,5
3,4,21,14423.46,300,4076.83,6,18,5,1,5.86,...,0,1,0,0,0,0,0,6,2023,24
4,5,43,42845.50,512,5000.00,9,0,0,0,7.36,...,0,0,1,0,0,0,0,10,2024,19


# Machine Learning Process

In [4]:
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier,
    AdaBoostClassifier
)
from sklearn.neural_network import MLPClassifier

from xgboost import XGBClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

In [19]:
df.head()

,user_id,age,monthly_income,credit_score,purchase_amount,bnpl_installments,repayment_delay_days,missed_payments,default_flag,app_usage_frequency,...,location_Canada,location_Germany,location_India,location_UK,location_USA,customer_segment_Low Risk,customer_segment_Medium Risk,transaction_month,transaction_year,transaction_day
0,1,56,68529.50,552,5000.00,12,13,1,0,8.49,...,0,0,0,0,0,0,1,6,2023,10
1,2,19,7247.85,300,1073.23,12,13,1,0,3.09,...,0,0,0,0,1,0,0,10,2024,7
2,3,20,41582.26,471,5000.00,3,19,2,0,3.33,...,0,0,0,0,0,0,0,4,2023,5
3,4,21,14423.46,300,4076.83,6,18,5,1,5.86,...,0,1,0,0,0,0,0,6,2023,24
4,5,43,42845.50,512,5000.00,9,0,0,0,7.36,...,0,0,1,0,0,0,0,10,2024,19


In [12]:
#Features and Target
# Copy dataframe
df_ml = df.copy()

# Drop unnecessary columns
df_ml = df_ml.drop(columns=[
    'user_id',
    'transaction_date'
], errors='ignore')

# Features and target
X = df_ml.drop('default_flag', axis=1)
y = df_ml['default_flag']

In [13]:
#Train-test split
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.2, random_state=42, stratify=y)

# Models
models ={
    "logistic_regression" :Pipeline([
        ('scaler', StandardScaler()),
        ('model', LogisticRegression(max_iter=1000))]),
    "Random Forest" : RandomForestClassifier(n_estimators=100, random_state=42),
    "Gradient Boosting" : GradientBoostingClassifier(n_estimators=100, random_state=42),
    "AdaBoost" : AdaBoostClassifier(n_estimators=100, random_state=42),
    "XGBoost" : XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=4, random_state=42, eval_metric='logloss'),
    "Neural Network" : Pipeline([
        ('scaler', StandardScaler()),
        ('model', MLPClassifier(hidden_layer_sizes=(64,32,), max_iter=300, random_state=42))])
}

In [16]:
#Train and Evaluate
result=[]

for name,model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    if hasattr(model, "predict_proba"):
        y_proba = model.predict_proba(X_test)[:,1]
    else:
        y_proba = model.decision_function(X_test)
    result.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1-Score': f1_score(y_test, y_pred),
        'ROC-AUC': roc_auc_score(y_test, model.predict_proba(X_test)[:,1])
    })
results_df = pd.DataFrame(result)
results_df = results_df.sort_values(by='ROC-AUC', ascending=False)
print(results_df.sort_values(by='ROC-AUC', ascending=False))

c:\Users\62853\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:780: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (300) reached and the optimization hasn't converged yet.
  warnings.warn(


                 Model  Accuracy  Precision    Recall  F1-Score   ROC-AUC
4              XGBoost  0.729821   0.764331  0.445545  0.562940  0.782393
2    Gradient Boosting  0.733204   0.799065  0.423267  0.553398  0.780041
3             AdaBoost  0.739004   0.835000  0.413366  0.552980  0.776064
1        Random Forest  0.714355   0.704331  0.462871  0.558626  0.768572
0  logistic_regression  0.689222   0.604828  0.589109  0.596865  0.758480
5       Neural Network  0.648139   0.553908  0.508663  0.530323  0.712396


In [17]:
# Best Model Analysis
best_model_name=results_df.iloc[0]['Model']
best_model=models[best_model_name]

print(f"Best Model: {best_model_name}")
y_pred_best = best_model.predict(X_test)
y_prob_best = best_model.predict_proba(X_test)[:,1]

print("\nClassification Report:")
print(classification_report(y_test, y_pred_best))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_best))

print(f"ROC-AUC Score: {roc_auc_score(y_test, y_prob_best)}")

Best Model: XGBoost

Classification Report:
              precision    recall  f1-score   support

           0       0.72      0.91      0.80      1261
           1       0.76      0.45      0.56       808

    accuracy                           0.73      2069
   macro avg       0.74      0.68      0.68      2069
weighted avg       0.74      0.73      0.71      2069


Confusion Matrix:
[[1150  111]
 [ 448  360]]
ROC-AUC Score: 0.7823931580311083


In [18]:
# Save Best Model
import joblib
joblib.dump(best_model, 'best_bnpl_model.pkl')
print("Best model saved as 'best_bnpl_model.pkl'")

Best model saved as 'best_bnpl_model.pkl'
